# EMD + LMS-Based Schizophrenia Classification on Dataset 1 using Statistical and Entropy Features

This notebook implements a combined EMD + LMS preprocessing pipeline for schizophrenia detection from EEG signals on Dataset 1.

Each EEG recording is segmented into 5-second windows, decomposed into Intrinsic Mode Functions (IMFs) using Empirical Mode Decomposition, and then filtered using an LMS procedure with the Fp1 channel as reference.

Statistical and entropy-based features are extracted from the filtered IMFs and used for machine learning classification.

In [2]:
import mne
from glob import glob
import numpy as np
from PyEMD import EMD
from scipy.signal import hilbert, lfilter, welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from sklearn.svm import SVC
from sklearn.model_selection import KFold, cross_val_score

In [3]:
def load_eeg(file_path):
    raw = mne.io.read_raw_edf(file_path, preload=True)
    return raw.get_data()

def segment_signal(signal, segment_duration, fs):

    segment_length = int(segment_duration * fs)
    return [signal[:, i:i+segment_length]
            for i in range(0, signal.shape[1], segment_length)]

In [4]:
healthy_file_paths = sorted(glob("dataverse_files/h*.edf"))
schizo_file_paths  = sorted(glob("dataverse_files/s*.edf"))

healthy_signals = [load_eeg(file) for file in healthy_file_paths]
schizo_signals  = [load_eeg(file) for file in schizo_file_paths]

fs = 250
segment_duration = 5

healthy_segments = [segment_signal(sig, segment_duration, fs) for sig in healthy_signals]
schizo_segments  = [segment_signal(sig, segment_duration, fs) for sig in schizo_signals]

Extracting EDF parameters from /content/dataverse_files/h01.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 231249  =      0.000 ...   924.996 secs...
Extracting EDF parameters from /content/dataverse_files/h02.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 227499  =      0.000 ...   909.996 secs...
Extracting EDF parameters from /content/dataverse_files/h03.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 227499  =      0.000 ...   909.996 secs...
Extracting EDF parameters from /content/dataverse_files/h04.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 231249  =      0.000 ...   924.996 secs...
Extracting EDF parameters from /content/dataverse_files/h05.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 236249  

In [5]:
healthy_segments = [seg for subj in healthy_segments for seg in subj if seg.shape[1] > 0]
schizo_segments  = [seg for subj in schizo_segments for seg in subj if seg.shape[1] > 0]

In [ ]:
def lms_filter_1coeff(imf, ref, mu=0.005):

    n_times = len(imf)
    output = np.zeros(n_times)
    w = 0.0
    for i in range(n_times):
        e = imf[i] - w * ref[i]
        w = w + 2 * mu * e * ref[i]
        output[i] = imf[i] - e
    return output


In [ ]:
def lms_filter_fp1_allch(signal, fp1_index=0, mu=0.005):
    signal = np.asarray(signal)
    n_channels, n_times = signal.shape

    filtered = np.zeros_like(signal)

    filtered[fp1_index, :] = signal[fp1_index, :]

    for ch in range(n_channels):
        if ch == fp1_index:
            continue
        filtered[ch, :] = lms_filter_1coeff(
            imf=signal[ch, :],
            ref=signal[fp1_index, :],
            mu=mu
        )
    return filtered

In [8]:
healthy_segments_filtered = Parallel(n_jobs=-1)(
    delayed(lms_filter_fp1_allch)(seg, fp1_index=0, mu=0.005)
    for seg in healthy_segments
)
schizo_segments_filtered = Parallel(n_jobs=-1)(
    delayed(lms_filter_fp1_allch)(seg, fp1_index=0, mu=0.005)
    for seg in schizo_segments
)

In [8]:
def compute_energy_1d(signal_1d):
    return np.sum(signal_1d**2)

def compute_std_1d(signal_1d):
    return np.std(signal_1d)

def compute_shannon_entropy_1d(signal_1d, eps=1e-10):
    abs_sig = np.abs(signal_1d)
    sum_sig = np.sum(abs_sig) + eps
    probs = abs_sig / sum_sig
    return -np.sum(probs * np.log(probs + eps))

def extract_features_per_segment(signal_2d):

    n_channels, n_times = signal_2d.shape
    feats = []
    for ch in range(n_channels):
        x = signal_2d[ch, :]
        en = compute_energy_1d(x)
        sd = compute_std_1d(x)
        ent = compute_shannon_entropy_1d(x)
        feats.extend([en, sd, ent])
    return np.array(feats)

In [ ]:
healthy_features = [extract_features_per_segment(seg) for seg in healthy_segments_filtered]
schizo_features  = [extract_features_per_segment(seg) for seg in schizo_segments_filtered]

X = np.vstack([healthy_features, schizo_features])
y = np.array([0]*len(healthy_features) + [1]*len(schizo_features))  # 0 = healthy, 1 = schizo

**EMD+LMS**

## LMS Filtering on IMFs

After EMD decomposition, LMS adaptive filtering is applied to the IMFs.

The IMFs of the Fp1 channel are used as reference, and the corresponding IMFs of the other channels are filtered accordingly.

This combined strategy aims to improve signal representation before feature extraction.

In [ ]:
def apply_emd_to_segment(segment, fs, max_imf=5):

    segment = np.asarray(segment)
    n_channels, n_times = segment.shape

    imfs_all = np.zeros((n_channels, max_imf, n_times))

    for ch in range(n_channels):
        # Extraire le canal ch
        x_ch = segment[ch, :] 

        t = np.linspace(0, n_times/fs, num=n_times, endpoint=False)

        emd = EMD()
        emd.max_imf = max_imf
        imfs = emd.emd(x_ch, t) 

        if imfs.shape[0] < max_imf:
            pad_rows = max_imf - imfs.shape[0]
            pad_array = np.zeros((pad_rows, n_times))
            imfs = np.vstack([imfs, pad_array])
        elif imfs.shape[0] > max_imf:
            imfs = imfs[:max_imf, :]

        imfs_all[ch, :, :] = imfs

    return imfs_all

### IMF count

To obtain a consistent input representation across samples, the number of retained IMFs per channel is fixed using zero-padding or truncation.

In [10]:
max_imf = 7

healthy_imfs_list = Parallel(n_jobs=-1)(
    delayed(apply_emd_to_segment)(seg, fs, max_imf=max_imf)
    for seg in healthy_segments
)
schizo_imfs_list = Parallel(n_jobs=-1)(
    delayed(apply_emd_to_segment)(seg, fs, max_imf=max_imf)
    for seg in schizo_segments
)

In [ ]:
def lms_emd_fp1(imfs_all, fp1_index=0, mu=0.005):

    imfs_all = np.asarray(imfs_all)
    n_channels, max_imf, n_times = imfs_all.shape

    imfs_fp1 = imfs_all[fp1_index, :, :]

    filtered = np.zeros_like(imfs_all)
    filtered[fp1_index, :, :] = imfs_fp1

    for ch in range(n_channels):
        if ch == fp1_index:
            continue
        for i_imf in range(max_imf):
            filtered[ch, i_imf, :] = lms_filter_1coeff(
                imfs_all[ch, i_imf, :],
                imfs_fp1[i_imf, :],
                mu=mu
            )
    return filtered

In [ ]:
healthy_imfs_filtered = Parallel(n_jobs=-1)(
    delayed(lms_emd_fp1)(imf_array, fp1_index=0, mu=0.005)
    for imf_array in healthy_imfs_list
)
schizo_imfs_filtered = Parallel(n_jobs=-1)(
    delayed(lms_emd_fp1)(imf_array, fp1_index=0, mu=0.005)
    for imf_array in schizo_imfs_list
)

In [ ]:
def extract_features_imfs(imfs_filtered):
    n_channels, max_imf, n_times = imfs_filtered.shape
    feat_list = []
    for ch in range(n_channels):
        for i_imf in range(max_imf):
            x = imfs_filtered[ch, i_imf, :]
            en = compute_energy_1d(x)
            st = compute_std_1d(x)
            ent = compute_shannon_entropy_1d(x)
            feat_list.extend([en, st, ent])
    return np.array(feat_list)

In [ ]:
healthy_features = [extract_features_imfs(arr) for arr in healthy_imfs_filtered]
schizo_features  = [extract_features_imfs(arr) for arr in schizo_imfs_filtered]

X = np.vstack([healthy_features, schizo_features])
y = np.array([0]*len(healthy_features) + [1]*len(schizo_features))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# RandomForest
clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)
print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))
print("Accuracy =", accuracy_score(y_test, y_pred_rf))

# SVM
clf_svm = SVC(kernel='rbf', random_state=42)
clf_svm.fit(X_train, y_train)
y_pred_svm = clf_svm.predict(X_test)
print("=== SVM ===")
print(classification_report(y_test, y_pred_svm))
print("Accuracy =", accuracy_score(y_test, y_pred_svm))